# Six Attempts Notebook  
## Logical and stable phase-recovery experiment harness

This notebook is a **real experiment notebook** with six different reconstruction attempts, all implemented in one place so you can compare whether failures come from:

- bad code
- weak measurement diversity
- missing constraints
- over-strong priors
- optimization issues
- or actual information limits in the inverse problem

## The six attempts

1. **Attempt 1 — Old GS baseline**  
2. **Attempt 2 — Stronger dispersion gap**  
3. **Attempt 3 — Hard spectral anchor**  
4. **Attempt 4 — Soft spectral prior**  
5. **Attempt 5 — Multi-plane GS**  
6. **Attempt 6 — Hybrid AI + GS**

The point is not to declare one “correct” automatically.  
The point is to compare them under the same benchmark.


## What this notebook tests

If a strong-prior or anchored version works while the pure GS version fails, then the code is probably not the main problem.

If adding more measurement diversity helps, then the issue is likely in the information content of the data.

If hybrid AI + GS works better than GS alone, then the learned initializer is adding useful prior structure.


In [ ]:
# ============================================================
# 1. Imports
# ============================================================

import json
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8.8, 4.8)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11

GLOBAL_SEED = 7
rng = np.random.default_rng(GLOBAL_SEED)

print("NumPy:", np.__version__)
print("Torch available:", TORCH_AVAILABLE)
if TORCH_AVAILABLE:
    print("Torch:", torch.__version__)


In [ ]:
# ============================================================
# 2. Constants and helpers
# ============================================================

GHZ = 1e9
NS = 1e-9
PS = 1e-12

N = 2**14
df = 0.05 * GHZ
f = (np.arange(N) - N // 2) * df
w = 2 * np.pi * f

dt = 1.0 / (N * df)
t = (np.arange(N) - N // 2) * dt

def fftc(x):
    return np.fft.fftshift(np.fft.fft(np.fft.ifftshift(x)))

def ifftc(X):
    return np.fft.fftshift(np.fft.ifft(np.fft.ifftshift(X)))

def normalize_peak(x, eps=1e-12):
    peak = np.max(np.abs(x))
    return x / max(peak, eps)

def rmse(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    return float(np.sqrt(np.mean((a - b) ** 2)))

def correlation_coefficient(a, b):
    a = np.asarray(a).ravel().astype(float)
    b = np.asarray(b).ravel().astype(float)
    a = a - a.mean()
    b = b - b.mean()
    denom = np.sqrt(np.sum(a**2) * np.sum(b**2)) + 1e-12
    return float(np.sum(a * b) / denom)

def gaussian_kernel_1d(length=101, sigma=12):
    x = np.arange(length) - length // 2
    k = np.exp(-(x**2) / (2 * sigma**2))
    return k / np.sum(k)

def gaussian_smooth(x, sigma=12, length=101):
    k = gaussian_kernel_1d(length=length, sigma=sigma)
    return np.convolve(np.asarray(x, dtype=float), k, mode="same")

print("Frequency span (GHz):", (f[-1] - f[0]) / GHZ)
print("Time span (ns):", (t[-1] - t[0]) / NS)
print("dt (ps):", dt / PS)


In [ ]:
# ============================================================
# 3. Spectrum models
# ============================================================

def gaussian_dip(f, center_hz, width_hz, depth=0.85):
    return 1.0 - depth * np.exp(-((f - center_hz) / width_hz) ** 2)

def make_single_line_spectrum(f):
    return np.clip(gaussian_dip(f, center_hz=40 * GHZ, width_hz=2.2 * GHZ, depth=0.95), 0.0, None)

def make_three_line_spectrum(f):
    S = np.ones_like(f, dtype=float)
    centers = [34 * GHZ, 40 * GHZ, 46 * GHZ]
    widths = [1.8 * GHZ, 1.8 * GHZ, 1.8 * GHZ]
    depths = [0.72, 0.88, 0.72]
    for c, wghz, d in zip(centers, widths, depths):
        S *= gaussian_dip(f, center_hz=c, width_hz=wghz, depth=d)
    return np.clip(S, 0.0, None)

def make_random_three_dip_spectrum(f, rng):
    centers = np.sort(rng.uniform(28, 52, size=3)) * GHZ
    widths = rng.uniform(1.4, 2.6, size=3) * GHZ
    depths = rng.uniform(0.55, 0.9, size=3)
    S = np.ones_like(f, dtype=float)
    for c, wghz, d in zip(centers, widths, depths):
        S *= gaussian_dip(f, center_hz=c, width_hz=wghz, depth=d)
    return np.clip(S, 0.0, None)


In [ ]:
# ============================================================
# 4. Forward model
# ============================================================

def make_field_from_psd(S, spectral_phase=None):
    amp = np.sqrt(np.maximum(S, 0.0))
    if spectral_phase is None:
        spectral_phase = np.zeros_like(amp)
    return amp * np.exp(1j * spectral_phase)

def dispersion_transfer_function(w, phi2=0.0, phi3=0.0):
    phase = 0.5 * phi2 * w**2 + (1.0 / 6.0) * phi3 * w**3
    return np.exp(1j * phase)

def propagate_dispersion_freq(Ef, w, phi2=0.0, phi3=0.0):
    H = dispersion_transfer_function(w, phi2=phi2, phi3=phi3)
    return ifftc(Ef * H)

def propagate_time_plane_to_time_plane(et, w, phi2_from, phi2_to, phi3_from=0.0, phi3_to=0.0):
    Ef_plane = fftc(et)
    H = np.exp(1j * (0.5 * (phi2_to - phi2_from) * w**2 + (1.0/6.0) * (phi3_to - phi3_from) * w**3))
    return ifftc(Ef_plane * H)

def intensity(x):
    return np.abs(x) ** 2

def make_measurements_from_spectrum(S, phi2_list, phi3_list=None):
    if phi3_list is None:
        phi3_list = [0.0] * len(phi2_list)
    Ef = make_field_from_psd(S)
    traces = []
    for phi2, phi3 in zip(phi2_list, phi3_list):
        et = propagate_dispersion_freq(Ef, w, phi2=phi2, phi3=phi3)
        traces.append(normalize_peak(intensity(et)))
    return traces


In [ ]:
# ============================================================
# 5. Benchmark case
# ============================================================

S_case = make_three_line_spectrum(f)

phi2_old = [1.2e-22, 2.4e-22]
phi2_strong = [1.0e-22, 6.0e-22]
phi2_multi = [1.0e-22, 3.5e-22, 7.0e-22]

traces_old = make_measurements_from_spectrum(S_case, phi2_old)
traces_strong = make_measurements_from_spectrum(S_case, phi2_strong)
traces_multi = make_measurements_from_spectrum(S_case, phi2_multi)

fig, axs = plt.subplots(1, 3, figsize=(14, 4))
for i, tr in enumerate(traces_old):
    axs[0].plot(t / NS, tr, label=f"old {i+1}")
axs[0].set_xlim(0, 0.7)
axs[0].set_title("Attempt 1 setup")
axs[0].legend()

for i, tr in enumerate(traces_strong):
    axs[1].plot(t / NS, tr, label=f"strong {i+1}")
axs[1].set_xlim(0, 0.7)
axs[1].set_title("Attempts 2/3/4/6 setup")
axs[1].legend()

for i, tr in enumerate(traces_multi):
    axs[2].plot(t / NS, tr, label=f"multi {i+1}")
axs[2].set_xlim(0, 0.7)
axs[2].set_title("Attempt 5 setup")
axs[2].legend()

for ax in axs:
    ax.set_xlabel("Time (ns)")
    ax.set_ylabel("Normalized intensity")

plt.tight_layout()
plt.show()


## Attempt 1 — Old GS baseline


In [ ]:
# ============================================================
# 6. Attempt 1: old GS
# ============================================================

def gs_two_plane_old(I1, I2, w, phi2_1, phi2_2, n_iter=120, seed=0):
    rng = np.random.default_rng(seed)
    mag1 = np.sqrt(np.maximum(I1, 0.0))
    mag2 = np.sqrt(np.maximum(I2, 0.0))
    phase0 = rng.uniform(-np.pi, np.pi, size=mag1.shape)
    e1 = mag1 * np.exp(1j * phase0)

    history = {"err1": [], "err2": [], "update": []}
    prev = e1.copy()

    for _ in range(n_iter):
        e2_pred = propagate_time_plane_to_time_plane(e1, w, phi2_1, phi2_2)
        err2 = np.sqrt(np.mean((np.abs(e2_pred) - mag2) ** 2))
        e2 = mag2 * np.exp(1j * np.angle(e2_pred))

        e1_pred = propagate_time_plane_to_time_plane(e2, w, phi2_2, phi2_1)
        err1 = np.sqrt(np.mean((np.abs(e1_pred) - mag1) ** 2))
        e1 = mag1 * np.exp(1j * np.angle(e1_pred))

        upd = np.sqrt(np.mean(np.abs(e1 - prev) ** 2))
        prev = e1.copy()

        history["err1"].append(err1)
        history["err2"].append(err2)
        history["update"].append(upd)

    return e1, history

def recover_psd_from_plane_field(e_plane, w, phi2_plane=0.0):
    Ef_plane = fftc(e_plane)
    H_back = np.exp(-1j * 0.5 * phi2_plane * w**2)
    Ef0 = Ef_plane * H_back
    return normalize_peak(np.abs(Ef0) ** 2), Ef0

e1_old, hist_old = gs_two_plane_old(traces_old[0], traces_old[1], w, phi2_old[0], phi2_old[1], n_iter=120, seed=1)
S_old, _ = recover_psd_from_plane_field(e1_old, w, phi2_plane=phi2_old[0])


## Attempt 2 — Stronger dispersion gap


In [ ]:
# ============================================================
# 7. Attempt 2: stronger dispersion gap
# ============================================================

e1_strong, hist_strong = gs_two_plane_old(traces_strong[0], traces_strong[1], w, phi2_strong[0], phi2_strong[1], n_iter=120, seed=1)
S_strong, _ = recover_psd_from_plane_field(e1_strong, w, phi2_plane=phi2_strong[0])


## Attempt 3 — Hard spectral anchor


In [ ]:
# ============================================================
# 8. Attempt 3: hard spectral anchor
# ============================================================

def gs_hard_anchor(I1, I2, S_target, w, phi2_1, phi2_2, n_iter=120, seed=0):
    rng = np.random.default_rng(seed)
    mag1 = np.sqrt(np.maximum(I1, 0.0))
    mag2 = np.sqrt(np.maximum(I2, 0.0))
    target_amp = np.sqrt(np.maximum(S_target, 0.0))

    phase0 = rng.uniform(-np.pi, np.pi, size=mag1.shape)
    e1 = mag1 * np.exp(1j * phase0)

    history = {"err1": [], "err2": [], "update": []}
    prev = e1.copy()

    for _ in range(n_iter):
        e2_pred = propagate_time_plane_to_time_plane(e1, w, phi2_1, phi2_2)
        err2 = np.sqrt(np.mean((np.abs(e2_pred) - mag2) ** 2))
        e2 = mag2 * np.exp(1j * np.angle(e2_pred))

        e0_pred = propagate_time_plane_to_time_plane(e2, w, phi2_2, 0.0)
        E0_pred = fftc(e0_pred)
        E0 = target_amp * np.exp(1j * np.angle(E0_pred))

        e1_pred = ifftc(E0 * np.exp(1j * 0.5 * phi2_1 * w**2))
        err1 = np.sqrt(np.mean((np.abs(e1_pred) - mag1) ** 2))
        e1 = mag1 * np.exp(1j * np.angle(e1_pred))

        upd = np.sqrt(np.mean(np.abs(e1 - prev) ** 2))
        prev = e1.copy()

        history["err1"].append(err1)
        history["err2"].append(err2)
        history["update"].append(upd)

    S_rec = normalize_peak(np.abs(E0) ** 2)
    return e1, S_rec, history

e1_anchor, S_anchor, hist_anchor = gs_hard_anchor(traces_strong[0], traces_strong[1], S_case, w, phi2_strong[0], phi2_strong[1], n_iter=120, seed=1)


## Attempt 4 — Soft spectral prior


In [ ]:
# ============================================================
# 9. Attempt 4: soft spectral prior
# ============================================================

def soft_spectral_anchor(E0, lam=0.10, sigma=14, length=121):
    mag = np.abs(E0)
    phase = np.angle(E0)
    mag_prior = gaussian_smooth(mag, sigma=sigma, length=length)
    mag_prior = np.maximum(mag_prior, 0.0)
    E_prior = mag_prior * np.exp(1j * phase)
    return (1 - lam) * E0 + lam * E_prior

def better_phase_init_from_intensity(I):
    x = normalize_peak(I)
    x_smooth = gaussian_smooth(x, sigma=8, length=81)
    phase = np.unwrap(np.angle(np.exp(1j * np.pi * x_smooth)))
    return np.angle(np.exp(1j * phase))

def gs_soft_prior(I1, I2, S_ref, w, phi2_1, phi2_2, n_iter=120, lam=0.10):
    mag1 = np.sqrt(np.maximum(I1, 0.0))
    mag2 = np.sqrt(np.maximum(I2, 0.0))
    phase0 = better_phase_init_from_intensity(I1)
    e1 = mag1 * np.exp(1j * phase0)

    history = {"err1": [], "err2": [], "update": [], "psd_rmse": []}
    prev = e1.copy()

    for _ in range(n_iter):
        e2_pred = propagate_time_plane_to_time_plane(e1, w, phi2_1, phi2_2)
        err2 = np.sqrt(np.mean((np.abs(e2_pred) - mag2) ** 2))
        e2 = mag2 * np.exp(1j * np.angle(e2_pred))

        e0_pred = propagate_time_plane_to_time_plane(e2, w, phi2_2, 0.0)
        E0_pred = fftc(e0_pred)
        E0 = soft_spectral_anchor(E0_pred, lam=lam)

        e1_pred = ifftc(E0 * np.exp(1j * 0.5 * phi2_1 * w**2))
        err1 = np.sqrt(np.mean((np.abs(e1_pred) - mag1) ** 2))
        e1 = mag1 * np.exp(1j * np.angle(e1_pred))

        upd = np.sqrt(np.mean(np.abs(e1 - prev) ** 2))
        prev = e1.copy()

        S_rec = normalize_peak(np.abs(E0) ** 2)
        history["err1"].append(err1)
        history["err2"].append(err2)
        history["update"].append(upd)
        history["psd_rmse"].append(rmse(normalize_peak(S_ref), S_rec))

    S_rec = normalize_peak(np.abs(E0) ** 2)
    return e1, S_rec, history

e1_soft, S_soft, hist_soft = gs_soft_prior(traces_strong[0], traces_strong[1], S_case, w, phi2_strong[0], phi2_strong[1], n_iter=120, lam=0.10)


## Attempt 5 — Multi-plane GS


In [ ]:
# ============================================================
# 10. Attempt 5: multi-plane GS
# ============================================================

def gs_multiplane(intensity_list, phi2_list, w, n_iter=120, seed=0):
    mags = [np.sqrt(np.maximum(I, 0.0)) for I in intensity_list]
    rng = np.random.default_rng(seed)
    e_planes = [None] * len(mags)
    phase0 = rng.uniform(-np.pi, np.pi, size=mags[0].shape)
    e_planes[0] = mags[0] * np.exp(1j * phase0)

    history = {"update": [], "constraint_errors": [[] for _ in range(len(mags))]}
    prev = e_planes[0].copy()

    for _ in range(n_iter):
        for p in range(1, len(mags)):
            pred = propagate_time_plane_to_time_plane(e_planes[p-1], w, phi2_list[p-1], phi2_list[p])
            err = np.sqrt(np.mean((np.abs(pred) - mags[p]) ** 2))
            history["constraint_errors"][p].append(err)
            e_planes[p] = mags[p] * np.exp(1j * np.angle(pred))

        for p in range(len(mags) - 2, -1, -1):
            pred = propagate_time_plane_to_time_plane(e_planes[p+1], w, phi2_list[p+1], phi2_list[p])
            err = np.sqrt(np.mean((np.abs(pred) - mags[p]) ** 2))
            history["constraint_errors"][p].append(err)
            e_planes[p] = mags[p] * np.exp(1j * np.angle(pred))

        upd = np.sqrt(np.mean(np.abs(e_planes[0] - prev) ** 2))
        prev = e_planes[0].copy()
        history["update"].append(upd)

    S_rec, _ = recover_psd_from_plane_field(e_planes[0], w, phi2_plane=phi2_list[0])
    return e_planes[0], S_rec, history

e1_multi, S_multi, hist_multi = gs_multiplane(traces_multi, phi2_multi, w, n_iter=120, seed=1)


## Attempt 6 — Hybrid AI + GS


In [ ]:
# ============================================================
# 11. Attempt 6: hybrid AI + GS
# ============================================================

feature_len = 256
spec_len = 256

def downsample_1d(x, new_len):
    idx = np.linspace(0, len(x) - 1, new_len)
    return np.interp(idx, np.arange(len(x)), np.asarray(x, dtype=float))

def compact_to_spectrum(S_compact, full_len):
    return normalize_peak(np.interp(
        np.linspace(0, len(S_compact) - 1, full_len),
        np.arange(len(S_compact)),
        np.asarray(S_compact, dtype=float)
    ))

def pack_measurement_features(I1, I2, feature_len=256):
    f1 = downsample_1d(normalize_peak(I1), feature_len)
    f2 = downsample_1d(normalize_peak(I2), feature_len)
    return np.concatenate([f1, f2], axis=0)

def spectrum_to_compact(S, spec_len=256):
    return downsample_1d(normalize_peak(S), spec_len)

def build_dataset(n_samples=180, seed=7):
    rng_local = np.random.default_rng(seed)
    X, Y = [], []
    for _ in range(n_samples):
        S = make_random_three_dip_spectrum(f, rng_local)
        traces = make_measurements_from_spectrum(S, phi2_strong)
        X.append(pack_measurement_features(traces[0], traces[1], feature_len=feature_len))
        Y.append(spectrum_to_compact(S, spec_len=spec_len))
    return np.asarray(X), np.asarray(Y)

X_all, Y_all = build_dataset(n_samples=180, seed=GLOBAL_SEED)
n_train = 150
X_train, Y_train = X_all[:n_train], Y_all[:n_train]
X_test, Y_test = X_all[n_train:], Y_all[n_train:]

def fit_linear_ridge(X, Y, reg=1e-3):
    Xb = np.concatenate([X, np.ones((X.shape[0], 1))], axis=1)
    I = np.eye(Xb.shape[1])
    W = np.linalg.solve(Xb.T @ Xb + reg * I, Xb.T @ Y)
    return W

def predict_linear_ridge(X, W):
    Xb = np.concatenate([X, np.ones((X.shape[0], 1))], axis=1)
    return Xb @ W

W_lin = fit_linear_ridge(X_train, Y_train, reg=1e-2)
feat_case = pack_measurement_features(traces_strong[0], traces_strong[1], feature_len=feature_len)[None, :]
compact_ai = np.clip(predict_linear_ridge(feat_case, W_lin)[0], 0.0, None)
S_ai = compact_to_spectrum(compact_ai, len(f))

def gs_hybrid(I1, I2, w, phi2_1, phi2_2, ai_prior_full, n_iter=120, alpha=0.25):
    mag1 = np.sqrt(np.maximum(I1, 0.0))
    mag2 = np.sqrt(np.maximum(I2, 0.0))

    E0_init = np.sqrt(np.maximum(ai_prior_full, 0.0)) * np.exp(1j * np.zeros_like(ai_prior_full))
    e1_init = ifftc(E0_init * np.exp(1j * 0.5 * phi2_1 * w**2))
    e1 = mag1 * np.exp(1j * np.angle(e1_init))

    history = {"err1": [], "err2": [], "update": []}
    prev = e1.copy()

    for _ in range(n_iter):
        e2_pred = propagate_time_plane_to_time_plane(e1, w, phi2_1, phi2_2)
        err2 = np.sqrt(np.mean((np.abs(e2_pred) - mag2) ** 2))
        e2 = mag2 * np.exp(1j * np.angle(e2_pred))

        e0_pred = propagate_time_plane_to_time_plane(e2, w, phi2_2, 0.0)
        E0_pred = fftc(e0_pred)

        phase = np.angle(E0_pred)
        E0_ai = np.sqrt(np.maximum(ai_prior_full, 0.0)) * np.exp(1j * phase)
        E0 = (1 - alpha) * E0_pred + alpha * E0_ai

        e1_pred = ifftc(E0 * np.exp(1j * 0.5 * phi2_1 * w**2))
        err1 = np.sqrt(np.mean((np.abs(e1_pred) - mag1) ** 2))
        e1 = mag1 * np.exp(1j * np.angle(e1_pred))

        upd = np.sqrt(np.mean(np.abs(e1 - prev) ** 2))
        prev = e1.copy()

        history["err1"].append(err1)
        history["err2"].append(err2)
        history["update"].append(upd)

    S_rec, _ = recover_psd_from_plane_field(e1, w, phi2_plane=phi2_1)
    return e1, S_rec, history

e1_hyb, S_hyb, hist_hyb = gs_hybrid(traces_strong[0], traces_strong[1], w, phi2_strong[0], phi2_strong[1], S_ai, n_iter=120, alpha=0.25)


## Comparison


In [ ]:
# ============================================================
# 12. Comparison metrics
# ============================================================

rows = [
    {"attempt": "1_old_gs", "rmse": rmse(normalize_peak(S_case), normalize_peak(S_old)), "corr": correlation_coefficient(normalize_peak(S_case), normalize_peak(S_old))},
    {"attempt": "2_stronger_gap", "rmse": rmse(normalize_peak(S_case), normalize_peak(S_strong)), "corr": correlation_coefficient(normalize_peak(S_case), normalize_peak(S_strong))},
    {"attempt": "3_hard_anchor", "rmse": rmse(normalize_peak(S_case), normalize_peak(S_anchor)), "corr": correlation_coefficient(normalize_peak(S_case), normalize_peak(S_anchor))},
    {"attempt": "4_soft_prior", "rmse": rmse(normalize_peak(S_case), normalize_peak(S_soft)), "corr": correlation_coefficient(normalize_peak(S_case), normalize_peak(S_soft))},
    {"attempt": "5_multiplane", "rmse": rmse(normalize_peak(S_case), normalize_peak(S_multi)), "corr": correlation_coefficient(normalize_peak(S_case), normalize_peak(S_multi))},
    {"attempt": "6_hybrid_ai", "rmse": rmse(normalize_peak(S_case), normalize_peak(S_hyb)), "corr": correlation_coefficient(normalize_peak(S_case), normalize_peak(S_hyb))},
]
rows


In [ ]:
# ============================================================
# 13. Main comparison figure
# ============================================================

fig, axs = plt.subplots(2, 2, figsize=(13, 8))

axs[0, 0].plot(hist_old["err1"], label="Attempt 1")
axs[0, 0].plot(hist_strong["err1"], label="Attempt 2")
axs[0, 0].plot(hist_anchor["err1"], label="Attempt 3")
axs[0, 0].plot(hist_soft["err1"], label="Attempt 4")
axs[0, 0].plot(hist_hyb["err1"], label="Attempt 6")
axs[0, 0].set_yscale("log")
axs[0, 0].set_title("Two-plane convergence")
axs[0, 0].legend()

for i, errs in enumerate(hist_multi["constraint_errors"]):
    if len(errs) > 0:
        axs[0, 1].plot(errs, label=f"plane {i+1}")
axs[0, 1].plot(hist_multi["update"], label="update", lw=1.8)
axs[0, 1].set_yscale("log")
axs[0, 1].set_title("Attempt 5 multi-plane convergence")
axs[0, 1].legend()

axs[1, 0].plot(f / GHZ, normalize_peak(S_case), "k--", lw=2, label="Ground truth")
axs[1, 0].plot(f / GHZ, normalize_peak(S_old), label="1 old")
axs[1, 0].plot(f / GHZ, normalize_peak(S_strong), label="2 gap")
axs[1, 0].plot(f / GHZ, normalize_peak(S_anchor), label="3 anchor")
axs[1, 0].plot(f / GHZ, normalize_peak(S_soft), label="4 soft")
axs[1, 0].plot(f / GHZ, normalize_peak(S_multi), label="5 multi")
axs[1, 0].plot(f / GHZ, normalize_peak(S_hyb), label="6 hybrid")
axs[1, 0].set_xlim(15, 80)
axs[1, 0].set_title("Recovered spectra")
axs[1, 0].legend(ncol=2, fontsize=9)

axs[1, 1].axis("off")
metric_text = "\n".join([f"{r['attempt']}: rmse={r['rmse']:.4f}, corr={r['corr']:.4f}" for r in rows])
axs[1, 1].text(0.02, 0.98, metric_text, va="top", family="monospace", fontsize=10)

fig.suptitle("Six-attempt logical/stable comparison harness")
plt.tight_layout()
plt.show()


## Interpretation guide

### If Attempt 3 works but Attempt 1 fails
That suggests the code path is mostly fine and the pure inverse problem is underconstrained.

### If Attempt 2 helps
That suggests measurement diversity matters.

### If Attempt 5 helps
That suggests more planes add useful information.

### If Attempt 6 helps
That suggests AI is adding a useful prior, not just random complexity.

This is the actual logic experiment you wanted.


In [ ]:
# ============================================================
# 14. Save summary
# ============================================================

summary = {
    "rows": rows,
    "torch_available": TORCH_AVAILABLE,
}

out_dir = Path("/mnt/data/six_attempts_outputs")
out_dir.mkdir(parents=True, exist_ok=True)

summary_path = out_dir / "six_attempts_summary.json"
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved summary to:", summary_path)


## Suggested commit message

```text
feat: add six-attempt phase-recovery comparison notebook

- implement six reconstruction attempts under a shared benchmark
- compare old GS, stronger gap, hard anchor, soft prior, multi-plane, and hybrid AI+GS
- add common metrics and convergence plots
- save experiment summary for reproducibility
```
